# Week4_ex4 - Lorentz Force Between Two Perpendicular Wires

Magnetostatic model of two straight wires at right angles - checked the Lorentz force on each wire, then isolated each wire's current to see the B-field it produces at the other wire's location.

In [ ]:
import pandas as pd
import numpy as np
import os
import ansys.aedt.core
import math
import shutil
import matplotlib.pyplot as plt

In [ ]:
# 1. AEDT desktop interface
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 2. turn off autosave
DT.disable_autosave()

# 3. create the project and set the solution type
sol_type = "Magnetostatic"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)

# 4. odesign object, needed to reuse script-recording snippets
oDesign = M3D.odesign


In [ ]:
## Set up output directory for results ##

proj_name = "Week4_ex4"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)

desi_name = "Week4_ex4"
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()


In [ ]:
# helper for finding coil terminal faces (kept from the source notebook - not actually used below,
# since this exercise picks terminals directly via bottom_face_x/top_face_x etc, but leaving it here for reference)
# uses each face's center x-coordinate: the two faces furthest out on the x-axis are the terminals
def find_terminal_face(winding_obj) :
    terminal_face = []

    # find maximum x position of winding object
    max_x_pos = max(abs(face.center[0]) for face in winding_obj.faces)

    # append terminal face to array
    for face in winding_obj.faces :
        if abs(abs(face.center[0]) - max_x_pos) <= 0.0001 :
            terminal_face.append(face)

    # sort
    ter_out, ter_in = sorted(terminal_face, key=lambda x : x.center[0], reverse=True)

    return ter_out, ter_in


In [ ]:
line_length = 500
M3D["line_length"] = f"{line_length}mm"

line_diameter = 10
M3D["line_diameter"] = f"{line_diameter}mm"

line_distance = 50
M3D["line_distance"] = f"{line_distance}mm"

line_num_seg = 12
M3D["line_num_seg"] = f"{line_num_seg}"

Current = 100
M3D["Current"] = f"{Current}A"


In [ ]:
## Create and configure analysis setup ##

my_setup = M3D.create_setup(name="Setup1")
my_setup.props["MaximumPasses"] = 20
my_setup.props["MinimumPasses"] = 2
my_setup.props["MinimumConvergedPasses"] = 1
my_setup.props["PercentRefinement"] = 15
my_setup.props["SolveFieldOnly"] = False
my_setup.props["PercentError"] = 1
my_setup.props["SolveMatrixAtLast"] = True
my_setup.props["UseNonLinearIterNum"] = False
my_setup.props["CacheSaveKind"] = "Delta"
my_setup.props["ConstantDelta"] = "0s"
my_setup.props["UseIterativeSolver"] = False
my_setup.props["RelativeResidual"] = 1E-06
my_setup.props["NonLinearResidual"] = 0.001
my_setup.props["RelaxationFactor"] = 1
my_setup.props["SmoothBHCurve"] = False
my_setup.props["MuOption"] = {"MuNonLinearBH": True}  # verify locally - same nested option as the raw InsertSetup version
my_setup.update()


In [ ]:
# 1. draw the straight wire
point_tmp = []
point_tmp.append(["-line_length/2", 0, 0])
point_tmp.append(["line_length/2", 0, 0])
line_1 = M3D.modeler.create_polyline(points=point_tmp, name="line_1", material="copper", xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)


# 2. draw a line to sample the field along
field_line_1 = M3D.modeler.create_polyline(points=point_tmp, name="field_line_1")


# 3. repeat 1-2 for the perpendicular wire
point_tmp = []
point_tmp.append([0, "-line_length/2", "line_distance"])
point_tmp.append([0, "line_length/2", "line_distance"])
line_2 = M3D.modeler.create_polyline(points=point_tmp, name="line_2", material="copper", xsection_type="Circle", xsection_width=line_diameter, xsection_num_seg=line_num_seg)

field_line_2 = M3D.modeler.create_polyline(points=point_tmp, name="field_line_2")


# 4. create the region
pad_value = ["line_length/2", "-line_length/2", "line_length/2", "-line_length/2", "line_length+line_distance", "-line_length"]
region = M3D.modeler.create_region(pad_value=pad_value, pad_type='Absolute Position', name='Region')


# 5. dummy objects, just to get a finer mesh for the plot
origin = ["-line_length/2", "-line_length/2", "-line_length/2"]
sizes = ["line_length", "line_length", "line_length+line_distance"]
dummy = M3D.modeler.create_box(origin=origin, sizes=sizes, name="dummy", material=None)

origin = ["-line_length/10", "-line_length/10", "-line_length/10"]
sizes = ["line_length/5", "line_length/5", "line_length/5+line_distance"]
dummy_2 = M3D.modeler.create_box(origin=origin, sizes=sizes, name="dummy_2", material=None)

M3D.modeler.subtract(blank_list=[dummy_2], tool_list=[line_1, line_2], keep_originals=True)


In [ ]:
# NOTE: MaxLength coarsened (line_length/12 -> line_length/4, and the dummy_2
# region from line_length/5/12 -> line_length/5/4), and PercentRefinement dropped
# from 30 to 15 -- at the original settings this pushed the mesh to ~195,822
# elements, well over the Student license's cap. Coarser mesh + slower per-pass
# refinement keeps it under the cap while still smoothing the field plot.
oModule = oDesign.GetModule("MeshSetup")

oModule.AssignLengthOp(
	[
		"NAME:Length1",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/4"
	])

oModule.AssignLengthOp(
	[
		"NAME:Length2",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["line_1","line_2"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/4"
	])

oModule.AssignLengthOp(
	[
		"NAME:Length3",
		"RefineInside:="	, True,
		"Enabled:="		, True,
		"Objects:="		, ["dummy_2"],
		"RestrictElem:="	, False,
		"NumMaxElem:="		, "1000",
		"RestrictLength:="	, True,
		"MaxLength:="		, "line_length/5/4"
	])



# M3D.mesh.generate_mesh(my_setup.name)

In [ ]:
# 1. set the coil terminal faces on each wire
ter_line1_in = line_1.bottom_face_x
ter_line1_out = line_1.top_face_x

ter_line2_in = line_2.bottom_face_y
ter_line2_out = line_2.top_face_y

M3D.assign_coil(assignment=ter_line1_in, conductors_number=1, polarity="Positive", name="in_1")
M3D.assign_coil(assignment=ter_line1_out, conductors_number=1, polarity="Negative", name="out_1")

M3D.assign_coil(assignment=ter_line2_in, conductors_number=1, polarity="Positive", name="in_2")
M3D.assign_coil(assignment=ter_line2_out, conductors_number=1, polarity="Negative", name="out_2")


# 2. create a winding for each wire and add its coil terminals
winding_1 = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                               resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_1")

winding_2 = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=True, current="Current", 
                               resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="winding_2")

M3D.add_winding_coils(assignment=winding_1.name, coils=["in_1", "out_1"])
M3D.add_winding_coils(assignment=winding_2.name, coils=["in_2", "out_2"])


In [ ]:
M3D.assign_force(assignment=line_1, coordinate_system='Global', is_virtual=False, force_name="force_line1")

M3D.assign_force(assignment=line_2, coordinate_system='Global', is_virtual=False, force_name="force_line2")


In [ ]:
my_setup.analyze()


In [ ]:
## force report ##

oModule = oDesign.GetModule("ReportSetup")

oModule.CreateReport("Force Table 1", "Magnetostatic", "Data Table", "Setup1 : LastAdaptive", [], 
	[
		"line_length:="		, ["All"],
		"line_diameter:="	, ["Nominal"],
		"line_distance:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "line_length",
		"Y Component:="		, ["force_line1.Force_x","force_line1.Force_y","force_line1.Force_z","force_line2.Force_x","force_line2.Force_y","force_line2.Force_z"]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line1.Force_x:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line1.Force_y:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line1.Force_z:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line2.Force_x:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line2.Force_y:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])

oModule.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Data Filter",
			[
				"NAME:PropServers", 
				"Force Table 1:force_line2.Force_z:Curve1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Units",
					"Value:="		, "newton"
				]
			]
		]
	])


csv_name = f"{proj_name}_force_table.csv"
dir_csv = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile("Force Table 1", dir_csv, False)



In [ ]:
## load the exported force data ##

# show 3 decimal places
pd.options.display.float_format = '{:.3f}'.format

# read the csv
csv_name = "Week4_ex4_force_table.csv"
df = pd.read_csv(csv_name)  # flat, next to the notebook -- matches where it was written above

df_filtered_line1 = df[['force_line1.Force_x [newton]', 'force_line1.Force_y [newton]', 'force_line1.Force_z [newton]']]
df_filtered_line2 = df[['force_line2.Force_x [newton]', 'force_line2.Force_y [newton]', 'force_line2.Force_z [newton]']]

display(df_filtered_line1)
display(df_filtered_line2)


In [ ]:
## check B_2 - the field that wire 2 creates at wire 1's location ##

M3D.cleanup_solution(variations='All', entire_solution=True, field=True, mesh=True, linked_data=True)

# turn off wire 1's current

winding_1["Current"] = "0"
display(winding_1.props)
winding_2["Current"] = "Current"
display(winding_2.props)


# solve

my_setup.analyze()


# save the B_2 data

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("B_2", "Fields", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Context:="		, "field_line_1",
		"PointCount:="		, 501
	], 
	[
		"Distance:="		, ["All"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_distance:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "Distance",
		"Y Component:="		, ["Mag_B"]
	])

csv_name = "Week4_ex4_B2.csv"
dir_csv = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile("B_2", dir_csv, False)


In [ ]:
## check B_1 - the field that wire 1 creates at wire 2's location ##

M3D.cleanup_solution(variations='All', entire_solution=True, field=True, mesh=True, linked_data=True)


# turn off wire 2's current

winding_1["Current"] = "Current"
display(winding_1.props)
winding_2["Current"] = "0"
display(winding_2.props)


# solve

my_setup.analyze()


# save the B_1 data

oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport("B_1", "Fields", "Rectangular Plot", "Setup1 : LastAdaptive", 
	[
		"Context:="		, "field_line_2",
		"PointCount:="		, 501
	], 
	[
		"Distance:="		, ["All"],
		"line_length:="		, ["Nominal"],
		"line_diameter:="	, ["Nominal"],
		"line_distance:="	, ["Nominal"],
		"line_num_seg:="	, ["Nominal"],
		"Current:="		, ["Nominal"]
	], 
	[
		"X Component:="		, "Distance",
		"Y Component:="		, ["Mag_B"]
	])

csv_name = "Week4_ex4_B1.csv"
dir_csv = csv_name  # saved flat, next to the notebook -- same convention as Images/
oModule.ExportToFile("B_1", dir_csv, False)


In [ ]:
## plot the B-field comparison ##

# load the data

b1_df = pd.read_csv(f"{proj_name}_B1.csv")  # flat -- matches where it was written above
b2_df = pd.read_csv(f"{proj_name}_B2.csv")  # flat -- matches where it was written above


# grab the column headers

head_b1 = b1_df.columns
head_b1 = list(head_b1)

head_b2 = b1_df.columns
head_b2 = list(head_b2)


# pull the units out of the header text

unit_len = head_b1[0].replace("Distance [", "")
unit_len = unit_len.replace("]", "")
unit_len

unit_T = head_b1[1].replace("Mag_B [", "")
unit_T = unit_T.replace("]", "")
unit_T


# pull out the data

length_data = b1_df[head_b1[0]].values
b1_data = b1_df[head_b1[1]].values
b2_data = b2_df[head_b2[1]].values

# average value and the theoretical value

avg_b1 = np.mean(b1_data)
avg_b2 = np.mean(b2_data)

mu0 = 4.0 * np.pi * 1e-7  # [H/m] permeability of free space

b1_formula = mu0*Current/( 2*math.pi*(line_distance*1e-3) ) * 1e+6  # uTesla
b2_formula = mu0*Current/( 2*math.pi*(line_distance*1e-3) ) * 1e+6  # uTesla


# plot

plt.figure(figsize=(8,6))
plt.plot(length_data, b1_data, 'r-', linewidth=5, label='B_1')
plt.plot(length_data, b2_data, 'b--', linewidth=5, label='B_2')
plt.xlabel(f'line position ({unit_len})')
plt.ylabel(f'B ({unit_T})')
plt.title(' ')
plt.legend()
plt.grid(True)
plt.ylim(100, 450)

os.makedirs("Images", exist_ok=True)
plt.savefig("Images/Week4_ex4_B_field_vs_position.png")
plt.show()

In [ ]:
M3D.save_project()
